# TV6 model comparison
Notebook này chỉ đọc các `metrics.json` thật do `scripts/evaluate.py` tạo. Model selection chính thức phải chạy bằng `scripts/compare_models.py` để tạo manifest bất biến.

In [ ]:
from pathlib import Path
import json
import pandas as pd

metric_paths = []  # Điền đường dẫn tới validation metrics.json thật
if not metric_paths:
    raise RuntimeError('Cần metric_paths từ các validation run thật; không dùng dữ liệu giả')
payloads = [json.loads(Path(path).read_text(encoding='utf-8')) for path in metric_paths]
if any(item['split'] != 'validation' for item in payloads):
    raise ValueError('Model comparison chỉ dùng validation')
if len({item['population_id'] for item in payloads}) != 1:
    raise ValueError('Các model không cùng evaluation population')
comparison = pd.DataFrame([{
    'run_id': item['run_id'], 'model_name': item['model_name'],
    'mae_deg_c': item['overall']['mae'], 'rmse_deg_c': item['overall']['rmse']
} for item in payloads]).sort_values('rmse_deg_c')
comparison